# 2. Tool calling, with no framework

**LangGraph tutorial, lesson 2 of 5**

The single most useful thing to understand about tool-calling agents:

> ## The model never runs any code.

It only ever emits text. When we say "the model called the calculator", what
physically happened is:

1. We sent the model a list of tool descriptions alongside the question.
2. The model replied with a **structured request**: a name and arguments.
3. **Our Python code** read that request and ran the function.
4. We sent the return value back as a new message.
5. The model read the result and wrote its final answer.

**Steps 3 and 4 are yours to write.** Skip them and the model just sits there
having asked for something it never receives.

This notebook does all five by hand. No LangGraph yet.

In [ ]:
from common import make_llm, text_of
from tools import ALL_TOOLS, calculator

for t in ALL_TOOLS:
    print(f"{t.name:18} {t.description.splitlines()[0]}")

## What a tool actually is

A tool is a plain Python function plus a description. The `@tool` decorator
(see `tools.py`) turns it into something the model can be told about.

Two things get sent to the model, so **both matter enormously**:

- the function **name**
- the **docstring** and type hints

That is the entire specification the model receives. A vague docstring is the
most common reason an agent picks the wrong tool or passes bad arguments.
**Write docstrings for the model, not for yourself.**

In [ ]:
import json

print("Name:", calculator.name)
print("Description:", calculator.description)
print("Args schema:", json.dumps(calculator.args, indent=2))

## Step 1: tell the model the tools exist

`bind_tools` does not change the model. It returns a copy that attaches the
tool schemas to every request.

In [ ]:
llm = make_llm()
llm_with_tools = llm.bind_tools(ALL_TOOLS)

## Step 2: the model requests a tool call

Note what does **not** happen below: no arithmetic. The model asks *us* to do it.

In [ ]:
question = "What is 4,871,203 times 9,284?"
first_reply = llm_with_tools.invoke(question)

print("Any text?  ", repr(text_of(first_reply)))
print("Tool calls:", first_reply.tool_calls)

That `tool_calls` list is the whole mechanism. The model has paused mid-thought
to wait for data. **Nothing has been computed yet** — if we stopped here, the
user gets nothing.

## Steps 3 & 4: our code runs the tool and returns the result

The result must go back as a `ToolMessage` carrying the matching
`tool_call_id`. That id is how the model pairs an answer to its question — get
it wrong and the API rejects the request.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

tools_by_name = {t.name: t for t in ALL_TOOLS}

# The conversation so far. This list is the agent's ENTIRE memory -- there is no
# hidden server-side state. Every turn re-sends the whole history.
conversation = [HumanMessage(question), first_reply]

for request in first_reply.tool_calls:
    chosen = tools_by_name[request["name"]]
    result = chosen.invoke(request["args"])          # <-- real Python runs here

    print(f"model asked : {request['name']}({request['args']})")
    print(f"python gave : {result}")

    conversation.append(
        ToolMessage(content=str(result), tool_call_id=request["id"])
    )

## Step 5: back to the model, now with the answer in hand

In [ ]:
second_reply = llm_with_tools.invoke(conversation)

print("Final answer:", text_of(second_reply))
print("Tool calls  :", second_reply.tool_calls, "<- empty, so we are done")

## The pattern to notice

We made **two** model calls for one question:

```
call 1  ->  "run calculator(4871203 * 9284)"
call 2  ->  "The answer is 45,224,248,652."
```

We stopped because call 2 had no `tool_calls`. But what if it had? We would need
to run those too, ask again, check again — **a loop whose length nobody knows in
advance.**

Lesson 3 writes that loop.

---
**Next:** `03_manual_loop.ipynb`